In [1]:
import openai
from PIL import Image
import requests

In [4]:
client = openai.Client()

### Gerar Imagens

In [26]:
nome="robo_praia"
model="gpt-image-1"
prompt="Crie uma imagem com dois robôs em uma praia lendo livro de android"
size="1024x1024"
quality="high"
style="vivid"

In [27]:
resposta = client.images.generate(
    model=model,
    prompt=prompt,
    size=size,
    quality=quality,
    n=1
)

In [28]:
resposta.data[0].revised_prompt

In [29]:
import base64

nome_arquivo = f"files/{nome}_{model}_{quality}_{style}.jpg"
image_b64 = resposta.data[0].b64_json

In [30]:
if image_b64 is None:
    print("None")
else :
    print(image_b64)

...


In [32]:
if image_b64 is not None:
    # 3. Decodifica o texto Base64 de volta para o formato de imagem (bytes)
    image_data = base64.b64decode(image_b64)
    
    # 4. Salva a imagem diretamente na sua pasta local
    with open(nome_arquivo, "wb") as f:
        f.write(image_data)
    print(f"Imagem salva com sucesso em: {nome_arquivo}")
else:
    print("Erro: A resposta não retornou os dados da imagem em b64_json.")

Imagem salva com sucesso em: files/robo_praia_gpt-image-1_high_vivid.jpg


### Image to Text

In [34]:
resposta = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{
        "role": "user",
        "content": [
            {"type": "text", "text": "Descreva a imagem fornecida"},
            {"type": "image_url", "image_url": {
                "url": "https://tse3.mm.bing.net/th/id/OIP.8t1Q8YjBbLVvS9z4SjQ9hAHaFB?r=0&rs=1&pid=ImgDetMain&o=7&rm=3"
            }}
        ]
    }]
)

In [9]:
resposta.choices[0].message.contents

'A imagem retrata uma praia ao pôr do sol, com um céu colorido em tons de laranja, rosa e lilás. À esquerda, há palmeiras altas que se inclinam em direção à água. O chão é coberto de areia fina, e a linha de água reflete as cores do céu. Ao fundo, pode-se ver uma pequena colina e vegetação, com algumas estruturas visíveis, possivelmente de restaurantes ou bares à beira-mar. A cena transmite uma atmosfera tranquila e tropical, ideal para relaxamento.'

### Extraia placa de carro

In [98]:
texto = "Qual é a placa? Devolva apenas a placa?"
txt_sys = """ Você é um validador de OCR de altíssima precisão especialista na legislação de trânsito e emplacamento do Brasil.
Sua missão é extrair o código alfanumérico da placa de um veículo e corrigir qualquer ambiguidade visual aplicando regras estruturais rígidas.

Siga estritamente o seguinte algoritmo de decisão passo a passo:

PASSO 1: IDENTIFICAÇÃO DO PADRÃO E CATEGORIA
Analise visualmente o formato da placa na imagem para determinar a máscara correta:
1. Padrão Antigo (Placa Cinza): Sempre 3 letras e 4 números. Máscara: [LLL-NNNN]
2. Padrão Mercosul (Carro): 3 letras, 1 número, 1 letra, 2 números. Máscara: [LLLNLNN]
3. Padrão Mercosul (Moto): 3 letras, 2 números, 1 letra, 1 número. Máscara: [LLLNNLN]

PASSO 2: APLICAÇÃO DE RESTRIÇÕES POSICIONAIS (DICIONÁRIO DE CORREÇÃO)
Após determinar o padrão no Passo 1, aplique rigorosamente as substituições de caracteres visualmente semelhantes:

- Onde a posição EXIGE uma LETRA:
  * Se houver '1', '|', '/' ou 'l' -> Converta para 'I'
  * Se houver '0' -> Converta para 'O'
  * Se houver '8' -> Converta para 'B'
  * Se houver '5' -> Converta para 'S'
  * Se houver '2' -> Converta para 'Z'
  * Se houver '6' -> Converta para 'G'

- Onde a posição EXIGE um NÚMERO:
  * Se houver 'I', 'l', '|' ou '/' -> Converta para '1'
  * Se houver 'O' ou 'Q' -> Converta para '0'
  * Se houver 'B' -> Converta para '8'
  * Se houver 'S' -> Converta para '5'
  * Se houver 'Z' -> Converta para '2'
  * Se houver 'G' -> Converta para '6'

PASSO 3: RESTRIÇÕES CRÍTICAS DE SAÍDA
- Ignore qualquer caractere ou ruído fora da área retangular da placa (como marcas d'água, parafusos ou textos do lacre).
- Remova qualquer hífen, espaço, ponto ou caractere especial.
- A saída deve conter EXATAMENTE 7 caracteres alfanuméricos em LETRAS MAIÚSCULAS.
- Se a imagem estiver totalmente ilegível, borrada ou sem placa visível, responda estritamente: "ERRO_LEITURA".

FORMATO DA RESPOSTA:
Retorne UNICAMENTE o código final da placa sem nenhum outro texto, saudação ou explicação (Exemplo: ABC1D23, AAA1111, XYZ12A3 ou ERRO_LEITURA).
"""

In [101]:
resposta = client.chat.completions.create(
    model="gpt-4o",
    messages=[
        {"role": "system", "content": txt_sys},
        {"role": "user",
        "content": [
            {"type": "text", "text": texto},
            {"type": "image_url", "image_url": {
                "url": "https://quatrorodas.abril.com.br/wp-content/uploads/2018/09/imagem-e1536873662306.jpg?quality=70&strip=info&w=720&crop=1"
            }}
        ]
    }],
    max_tokens=10,
    temperature=0.5
)

In [102]:
resposta.choices[0].message.content

'RI02A19'

### OCR

In [50]:
texto = "O que está escrito na imagem?"

In [51]:
resposta = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[{
        "role": "user",
        "content": [
            {"type": "text", "text": texto},
            {"type": "image_url", "image_url": {
                "url": "https://www.rbsdirect.com.br/imagesrc/17752154.jpg?w=700"
            }}
        ]
    }],
    max_tokens=1000
)

In [52]:
resposta.choices[0].message.content

'Está escrito:\n\n"ISSO É UM PEDIDO:\n\nQue você seja muito feliz no seu grande dia. Este vestido tem mãos de uma brasileira, gera emprego e movimenta a economia local.\n\nFoi um prazer fazê-lo para você.\n\nElianna\nbetina-mari,\nBelo Horizonte / 15."'

In [54]:
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": "What are in these images? Is there any difference between them?",
                },
                {
                    "type": "image_url",
                    "image_url": {
                        "url": "https://www.bing.com/th/id/OIP.GKvzXKtR4cO5ivJKtV4gKQHaEK?w=193&h=135&c=8&rs=1&qlt=90&o=6&dpr=1.3&pid=ImgAns&rm=2",
                    },
                },
                {
                    "type": "image_url",
                    "image_url": {
                        "url": "https://www.bing.com/th/id/OIP.2WDApmBBY5hkbhqAzlp8tAHaEK?w=193&h=135&c=8&rs=1&qlt=90&o=6&dpr=1.3&pid=ImgAns&rm=2",
                    },
                },
            ],
        }
    ],
    max_tokens=300,
)

In [55]:
response.choices[0].message.content

'The two images depict different natural scenes. \n\n1. The first image features a waterfall cascading over rocky cliffs into a green pond, surrounded by lush vegetation. It conveys a vibrant, tropical setting.\n\n2. The second image shows a serene beach scene at sunset, characterized by calm waters, a distant sun, and a palm tree silhouetted against the glowing sky. This scene evokes a tranquil and warm atmosphere.\n\nIn summary, the main differences are the types of water bodies (waterfall vs. ocean), the time of day (daytime vs. sunset), and the overall moods conveyed by each image (dynamic vs. calm).'